# train_gru_gcn_v3.ipynb**GRU-GCN** — 12-horizon, VRAM-safe for GTX 1650 4GB**Runs from:** `backend/model training/`

In [ ]:
import os, gc, pickle, time, warnings, mathimport numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as Fimport scipy.sparse as sp, matplotlib.pyplot as plt, seaborn as snsfrom pathlib import Pathfrom sklearn.metrics import mean_absolute_error, mean_squared_error, r2_scorewarnings.filterwarnings('ignore'); sns.set_theme(style='darkgrid')device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')print(f'Device: {device}')MODEL_NAME='gru_gcn'; MODEL_DISPLAY='GRU-GCN'LOOKBACK=24; N_HORIZONS=12; RAW_FEAT_DIM=33; EMBED_DIM=32; IN_DIM=65TRAIN_END=107; VAL_END=143; TEST_END=167LR=2e-3; WARMUP_EP=5; PATIENCE=15; MAX_EPOCHS=80; WEIGHT_DECAY=1e-4DATA_DIR='../data generation/data'PROC_DIR=f'{DATA_DIR}/processed'; SCALER_DIR=f'{DATA_DIR}/scalers'ADJ_DIR=f'{DATA_DIR}/graph_adj'; TENSOR_DIR=f'{DATA_DIR}/corridor_tensors'MODEL_DIR=f'../models/{MODEL_NAME}'os.makedirs(MODEL_DIR, exist_ok=True); os.makedirs(TENSOR_DIR, exist_ok=True)def _save(o,p): torch.save(o, str(Path(p).resolve()))def _load(p,**k): return torch.load(str(Path(p).resolve()), **k)with open(f'{SCALER_DIR}/scaler_ttr.pkl','rb') as f: scaler_ttr=pickle.load(f)with open(f'{SCALER_DIR}/scaler_cong.pkl','rb') as f: scaler_cong=pickle.load(f)with open(f'{SCALER_DIR}/feat_meta.pkl','rb') as f: feat_meta=pickle.load(f)assert feat_meta['RAW_FEAT_DIM']==RAW_FEAT_DIMwith open(f'{ADJ_DIR}/corridor_meta.pkl','rb') as f: corridor_meta=pickle.load(f)corridors=sorted(corridor_meta.keys())N_TOTAL_EDGES=sum(corridor_meta[c]['N'] for c in corridors)with open(f'{PROC_DIR}/corridor_batch_map.pkl','rb') as f: corridor_batch_map=pickle.load(f)df_static=pd.read_parquet(f'{PROC_DIR}/edges_static_scaled.parquet')fftt_lookup=df_static.set_index('edge_id')['free_flow_travel_time'].to_dict()del df_static; gc.collect()print(f'Model: {MODEL_DISPLAY}, Corridors: {len(corridors)}, Edges: {N_TOTAL_EDGES:,}')

In [ ]:
def build_corridor_tensors():    done = f'{TENSOR_DIR}/.done_v4'    if os.path.exists(done):        print(f'Corridor tensors cached → {TENSOR_DIR}/'); return    import glob as gl    for f in gl.glob(f'{TENSOR_DIR}/corridor_*.pt'): os.remove(f)    print('Building corridor tensors (batch-by-batch)...')    t0 = time.perf_counter()    batch_paths = sorted(gl.glob(f'{PROC_DIR}/batch_*.parquet'))    feat_cols = feat_meta['ALL_FEAT_COLS']    for cid in corridors:        meta = corridor_meta[cid]; N = meta['N']        tensor = np.zeros((N, 168, RAW_FEAT_DIM), dtype=np.float32)        local_map = meta['local_map']        for bi in corridor_batch_map.get(cid, []):            df = pd.read_parquet(batch_paths[bi])            cdf = df[df['corridor_id'] == cid]            for eid in meta['edge_ids']:                li = local_map.get(eid)                if li is None: continue                edf = cdf[cdf['edge_id'] == eid].sort_values('hour_index')                if len(edf) == 0: continue                tensor[li, edf['hour_index'].values, :] = edf[feat_cols].values.astype(np.float32)            del df, cdf; gc.collect()        _save(torch.from_numpy(tensor), f'{TENSOR_DIR}/corridor_{cid:04d}.pt')        del tensor; gc.collect()    with open(done, 'w') as f: f.write('v4')    print(f'Done in {time.perf_counter()-t0:.1f}s')build_corridor_tensors()

In [ ]:
def get_windows(split='train'):    if split=='train': return list(range(LOOKBACK-1, TRAIN_END-N_HORIZONS+1))    elif split=='val': return list(range(TRAIN_END-LOOKBACK+1, VAL_END-N_HORIZONS+1))    else: return list(range(VAL_END-LOOKBACK+1, TEST_END-N_HORIZONS+1))# GRU-GCN: 2 GCN layers per timestep × 24 steps + GRU(2 layers, hidden=192)# Memory per sample ≈ N × 120KB (lighter than GWN)MAX_N = 8000VRAM_ACT_BUDGET = 2.5e9MEM_PER_ELEM = 120_000def get_bs(N):    bs = max(1, int(VRAM_ACT_BUDGET / (N * MEM_PER_ELEM)))    return min(bs, 8)train_windows=get_windows('train'); val_windows=get_windows('val'); test_windows=get_windows('test')print(f'Windows: train={len(train_windows)} val={len(val_windows)} test={len(test_windows)}')print(f'VRAM safety: MAX_N={MAX_N}, bs examples: N=500→{get_bs(500)}, N=2000→{get_bs(2000)}, N=5000→{get_bs(5000)}')

In [ ]:
class GCNLay(nn.Module):    def __init__(s,ic,oc):        super().__init__(); s.lin=nn.Linear(ic,oc); s.nm=nn.LayerNorm(oc)        s.res=nn.Linear(ic,oc) if ic!=oc else nn.Identity()    def forward(s,x,A):        B,N,C=x.shape        if A.is_sparse:            h=torch.sparse.mm(A, x.reshape(B*N,C).T.reshape(N,B*C)).T.reshape(B,N,C)        else:            h=torch.bmm(A.unsqueeze(0).expand(B,-1,-1),x)        return s.nm(F.relu(s.lin(h))+s.res(x))class GRU_GCN(nn.Module):    def __init__(s,in_dim=IN_DIM,hidden=192,nh=N_HORIZONS):        super().__init__(); s.inp=nn.Linear(in_dim,hidden)        s.g1=GCNLay(hidden,hidden); s.g2=GCNLay(hidden,hidden)        s.gru=nn.GRU(hidden,hidden,num_layers=2,batch_first=True,dropout=0.1); s.fc=nn.Linear(hidden,nh)    def forward(s,x,adj):        B,N,T,_=x.shape; h=s.inp(x)        hs=[s.g2(s.g1(h[:,:,t,:],adj),adj) for t in range(T)]        h=torch.stack(hs,2).reshape(B*N,T,-1)        go,_=s.gru(h); return s.fc(go[:,-1,:]).reshape(B,N,-1)def build_model(): return GRU_GCN()

In [ ]:
def load_adj(cid,N):    with open(f'{ADJ_DIR}/corridor_{cid:04d}.pkl','rb') as f: ad=pickle.load(f)    r,c,v=ad['A_fwd_coo']    r2=np.concatenate([r,np.arange(N,dtype=np.int32)]); c2=np.concatenate([c,np.arange(N,dtype=np.int32)])    v2=np.concatenate([v,np.ones(N,dtype=np.float32)]); A=sp.coo_matrix((v2,(r2,c2)),shape=(N,N)).tocsr()    d=np.array(A.sum(1)).flatten(); d[d==0]=1; An=sp.diags(1/d)@A; Ac=An.tocoo()    i=torch.stack([torch.from_numpy(Ac.row.astype(np.int64)),torch.from_numpy(Ac.col.astype(np.int64))])    return torch.sparse_coo_tensor(i,torch.from_numpy(Ac.data.astype(np.float32)),(N,N)).coalesce()def train_one_target(tidx, tname, sname, edge_emb):    print(f'\n{"="*60}\n  {MODEL_DISPLAY} — {tname} (idx={tidx})\n{"="*60}')    model=build_model().to(device)    np_m=sum(p.numel() for p in model.parameters() if p.requires_grad)    print(f'  Params: {np_m:,} + embed: {sum(p.numel() for p in edge_emb.parameters()):,}')    all_p=list(model.parameters())+list(edge_emb.parameters())    opt=torch.optim.AdamW(all_p, lr=LR, weight_decay=WEIGHT_DECAY)    sched=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,'min',factor=0.5,patience=7,min_lr=1e-6)    crit=nn.SmoothL1Loss(); best=float('inf'); pat=0    hist={'train_loss':[],'val_loss':[],'lr':[]}    skipped=set()    for epoch in range(MAX_EPOCHS):        te=time.perf_counter()        for pg in opt.param_groups: pg['lr']=LR*min(1,(epoch+1)/WARMUP_EP)        model.train(); edge_emb.train(); eloss=0; ns=0        for cid in np.random.permutation(corridors):            mc=corridor_meta[cid]; N=mc['N']            if N<3 or N>MAX_N: continue            data=_load(f'{TENSOR_DIR}/corridor_{cid:04d}.pt',weights_only=True)            adj=load_adj(cid,N); adj=adj.to(device)            gi=torch.from_numpy(mc['global_indices']).long().to(device)            bs=get_bs(N); wi=np.random.permutation(train_windows)            try:                for s in range(0,len(wi),bs):                    bt=wi[s:min(s+bs,len(wi))]                    emb=edge_emb(gi)                    xl,yl=[],[]                    for t in bt:                        xf=data[:,t-LOOKBACK+1:t+1,:].to(device)                        xl.append(torch.cat([xf,emb.unsqueeze(1).expand(-1,LOOKBACK,-1)],dim=-1))                        yl.append(data[:,t+1:t+1+N_HORIZONS,tidx].to(device))                    x=torch.stack(xl); y=torch.stack(yl)                    del xl,yl,emb  # free intermediates before forward pass                    opt.zero_grad(); loss=crit(model(x,adj),y); loss.backward()                    torch.nn.utils.clip_grad_norm_(all_p,5.0); opt.step()                    eloss+=loss.item()*len(bt); ns+=len(bt)                    del x,y,loss; torch.cuda.empty_cache()  # free after EACH mini-batch            except RuntimeError as e:                if 'out of memory' in str(e).lower():                    if cid not in skipped:                        print(f'    OOM at corridor {cid} (N={N}), skipping'); skipped.add(cid)                    del data,adj,gi; torch.cuda.empty_cache(); gc.collect(); continue                raise            del data,adj,gi; torch.cuda.empty_cache()        tl=eloss/max(ns,1)        model.eval(); edge_emb.eval(); vs=0; vn=0        with torch.no_grad():            for cid in corridors:                mc=corridor_meta[cid]; N=mc['N']                if N<3 or N>MAX_N or cid in skipped: continue                data=_load(f'{TENSOR_DIR}/corridor_{cid:04d}.pt',weights_only=True)                adj=load_adj(cid,N); adj=adj.to(device)                gi=torch.from_numpy(mc['global_indices']).long().to(device)                emb=edge_emb(gi)                for t in val_windows:                    xf=data[:,t-LOOKBACK+1:t+1,:].to(device)                    x=torch.cat([xf,emb.unsqueeze(1).expand(-1,LOOKBACK,-1)],dim=-1).unsqueeze(0)                    y=data[:,t+1:t+1+N_HORIZONS,tidx].to(device).unsqueeze(0)                    vs+=crit(model(x,adj),y).item(); vn+=1                del data,adj,emb; torch.cuda.empty_cache()        vl=vs/max(vn,1); hist['train_loss'].append(tl); hist['val_loss'].append(vl); hist['lr'].append(opt.param_groups[0]['lr'])        if epoch>=WARMUP_EP: sched.step(vl)        mk=''        if vl<best: best=vl; pat=0; _save({'model':model.state_dict(),'embedding':edge_emb.state_dict()},f'{MODEL_DIR}/{sname}'); mk=' ★'        else: pat+=1        print(f'  Ep {epoch+1:3d} | trn={tl:.5f} val={vl:.5f} lr={opt.param_groups[0]["lr"]:.1e} | {time.perf_counter()-te:.0f}s{mk}')        if pat>=PATIENCE: print('  Early stop'); break        torch.cuda.empty_cache(); gc.collect()    if skipped: print(f'  Note: {len(skipped)} corridors skipped (OOM): N up to {max(corridor_meta[c]["N"] for c in skipped)}')    ck=_load(f'{MODEL_DIR}/{sname}',weights_only=True)    model.load_state_dict(ck['model']); edge_emb.load_state_dict(ck['embedding'])    return model, edge_emb, hist

In [ ]:
edge_emb_ttr=nn.Embedding(N_TOTAL_EDGES,EMBED_DIM).to(device)model_ttr, edge_emb_ttr, hist_ttr = train_one_target(0,'TTR','best_ttr.pt',edge_emb_ttr)model_ttr.cpu(); edge_emb_ttr.cpu(); torch.cuda.empty_cache(); gc.collect()edge_emb_cong=nn.Embedding(N_TOTAL_EDGES,EMBED_DIM).to(device)model_cong, edge_emb_cong, hist_cong = train_one_target(1,'Congestion','best_cong.pt',edge_emb_cong)model_ttr.to(device); edge_emb_ttr.to(device)print(f'\n✓ Both saved to {MODEL_DIR}/')

In [ ]:
def evaluate_model(model, emb, tidx, scaler):    model.to(device); emb.to(device)    model.eval(); emb.eval(); ap,aa,af=[],[],[]    with torch.no_grad():        for cid in corridors:            mc=corridor_meta[cid]; N=mc['N']            if N<3 or N>MAX_N: continue            data=_load(f'{TENSOR_DIR}/corridor_{cid:04d}.pt',weights_only=True)            adj=load_adj(cid,N); adj=adj.to(device)            gi=torch.from_numpy(mc['global_indices']).long().to(device)            e=emb(gi); ff=np.array([fftt_lookup.get(eid,30.) for eid in mc['edge_ids']],np.float32)            for t in test_windows:                xf=data[:,t-LOOKBACK+1:t+1,:].to(device)                x=torch.cat([xf,e.unsqueeze(1).expand(-1,LOOKBACK,-1)],dim=-1).unsqueeze(0)                y=data[:,t+1:t+1+N_HORIZONS,tidx]                ap.append(model(x,adj).squeeze(0).cpu().numpy()); aa.append(y.numpy())                af.append(np.tile(ff[:,None],(1,N_HORIZONS)))            del data,adj; torch.cuda.empty_cache()    p=np.concatenate(ap); a=np.concatenate(aa); f=np.concatenate(af)    return p*scaler.scale_[0]+scaler.mean_[0], a*scaler.scale_[0]+scaler.mean_[0], fttr_p,ttr_a,ttr_f=evaluate_model(model_ttr,edge_emb_ttr,0,scaler_ttr)cong_p,cong_a,_=evaluate_model(model_cong,edge_emb_cong,1,scaler_cong)ttr_ps=ttr_p*ttr_f; ttr_as=ttr_a*ttr_fprint(f'TTR: {ttr_p.shape}, Cong: {cong_p.shape}')

In [ ]:
def metrics(p,a,name):    r=[]    for h in range(N_HORIZONS):        pp,aa=p[:,h],a[:,h]; m=np.abs(aa)>0.1        r.append({'h':h+1,'MAE':mean_absolute_error(aa,pp),'RMSE':np.sqrt(mean_squared_error(aa,pp)),                  'R2':r2_score(aa,pp),'MAPE':np.mean(np.abs((aa[m]-pp[m])/aa[m]))*100 if m.sum()>0 else float('nan')})    df=pd.DataFrame(r); print(f'\n── {name} ──'); print(df.to_string(index=False,float_format='%.4f')); return dfmt=metrics(ttr_p,ttr_a,'TTR ratio'); ms=metrics(ttr_ps,ttr_as,'TTR sec'); mc=metrics(cong_p,cong_a,'Congestion')

In [ ]:
fig,ax=plt.subplots(3,3,figsize=(18,15)); fig.suptitle(f'{MODEL_DISPLAY}',fontsize=16,fontweight='bold')ax[0,0].plot(hist_ttr['train_loss'],label='TTR trn',color='steelblue'); ax[0,0].plot(hist_ttr['val_loss'],label='TTR val',color='steelblue',ls='--')ax[0,0].plot(hist_cong['train_loss'],label='Cng trn',color='coral'); ax[0,0].plot(hist_cong['val_loss'],label='Cng val',color='coral',ls='--')ax[0,0].legend(fontsize=8); ax[0,0].set_yscale('log'); ax[0,0].set_title('Loss')ns=min(10000,ttr_ps.size); ii=np.random.choice(ttr_ps.size,ns,replace=False)ax[0,1].scatter(ttr_as.flat[ii],ttr_ps.flat[ii],alpha=.1,s=3,c='steelblue'); ax[0,1].set_title('TTR(s)')ax[0,2].scatter(cong_a.flat[ii],cong_p.flat[ii],alpha=.1,s=3,c='coral'); ax[0,2].set_title('Cong')te=ttr_p.flat-ttr_a.flat; ce=cong_p.flat-cong_a.flatsns.histplot(te[np.abs(te)<np.percentile(np.abs(te),99)],kde=True,ax=ax[1,0],color='steelblue',bins=50)ax[1,1].scatter(ttr_p.flat[ii],te[ii],alpha=.1,s=3,c='steelblue'); ax[1,1].axhline(0,c='r',ls='--')ta=np.sort(np.abs(te)); ca=np.sort(np.abs(ce))ax[1,2].plot(ta,np.linspace(0,1,len(ta)),label='TTR'); ax[1,2].plot(ca,np.linspace(0,1,len(ca)),label='Cng')ax[1,2].set_xlim(0,np.percentile(ta,98)); ax[1,2].legend()xh=mt['h'].values; ax[2,0].bar(xh-.2,mt['MAE'],.35,label='TTR',color='steelblue'); ax[2,0].bar(xh+.2,mc['MAE'],.35,label='Cng',color='coral')ax[2,0].set_xticks(xh); ax[2,0].legend(); ax[2,0].set_title('MAE/Horizon')ch=max(1,ttr_as.shape[0]//len(test_windows)); nw=min(len(test_windows),20)ax[2,1].plot([ttr_as[i*ch:(i+1)*ch,0].mean() for i in range(nw)],'o-',label='Act',ms=4)ax[2,1].plot([ttr_ps[i*ch:(i+1)*ch,0].mean() for i in range(nw)],'s--',label='Pred',ms=4); ax[2,1].legend()ax[2,2].axis('off')s=[['','TTR','TTR s','Cng'],['MAE h1',f'{mt.iloc[0]["MAE"]:.4f}',f'{ms.iloc[0]["MAE"]:.1f}',f'{mc.iloc[0]["MAE"]:.4f}'],   ['MAE h12',f'{mt.iloc[11]["MAE"]:.4f}',f'{ms.iloc[11]["MAE"]:.1f}',f'{mc.iloc[11]["MAE"]:.4f}'],   ['R² h1',f'{mt.iloc[0]["R2"]:.4f}','',f'{mc.iloc[0]["R2"]:.4f}'],['R² h12',f'{mt.iloc[11]["R2"]:.4f}','',f'{mc.iloc[11]["R2"]:.4f}']]ax[2,2].table(cellText=s[1:],colLabels=s[0],loc='center',cellLoc='center').auto_set_font_size(False)plt.tight_layout(); plt.savefig(f'{MODEL_DIR}/eval.png',dpi=150,bbox_inches='tight'); plt.show()print(f'✓ {MODEL_DISPLAY} complete.')